# Figure 5b — Axoaxonic / axodendritic / dendritic output fractions

For each of the 34 PSI-IhNs, what fraction of its output synapses are axoaxonic (onto
sensory axons), axodendritic (from its axon onto other neurons' dendrites), or dendritic
(from its *dendrite* onto other neurons — dendrodendritic output). Bars are population
means, dots are individual neurons.

**This is the only Figure 5 panel that needs CAVE** — the counts are live synapse queries.

| Fraction | Table | Filter |
| --- | --- | --- |
| Axoaxonic | `axoaxonic_output_synapses` | `pre_pt_root_id` = the **axon** segment; no further filtering |
| Axodendritic | `synapses_version3` | **total axon output minus the axoaxonic count** — `pre_pt_root_id` = the **axon** segment (autapses dropped, cleft `size` ≥ 20), then subtract the axoaxonic synapses, which are also present in that table |
| Dendritic | `synapses_version3` | `pre_pt_root_id` = the **dendrite** segment; same filters |

Each PSI-IhN was manually split into axon and dendrite compartments, which exist as two
separate segments. `../data/psi_ihn_compartments.csv` holds the seed coordinate and
resolved root ID of each compartment, so no CloudVolume lookup is needed here.

> ### Use a materialization **later than 669** for this panel
>
> The axodendritic and dendritic fractions are only meaningful where a neuron's axon and
> dendrite are genuinely separate segments. In materialization **669** — the newest version
> available when this repository was assembled — **2 of the 34 neurons** (both
> `islet-cltmr`, seeds `22476_4688_1997` and `24266_4298_1651`) had both compartment
> coordinates resolving to a single merged segment. Their dendritic count was then just the
> whole neuron's output counted twice (3427 + 149 = 3576 and 2775 + 155 = 2930 exactly),
> pinning them near 0.50/0.50 and distorting the `islet-cltmr` mean.
>
> **These two neurons have since been split manually**, so any materialization newer than
> 669 should contain the corrected segmentation. To use it:
>
> 1. re-run `data_generation/build_psi_ihn_compartment_map.ipynb` against the new version —
>    it rewrites `../data/psi_ihn_compartments.csv` with the new root IDs;
> 2. set `MATERIALIZATION` below to that version;
> 3. confirm the cell after it reports **0 unsplit neurons**.
>
> Until then the notebook plots the panel twice — with all 34 neurons, and with the 2
> excluded. Excluding them brings `islet-cltmr` (0.096 / 0.131 / 0.773) essentially on top
> of `islet-chtmr` (0.096 / 0.136 / 0.768), which is the expected result for two islet
> populations, so the split-only version is the better estimate of the published panel.


In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "svg.fonttype": "none",
    "font.size": 6, "axes.titlesize": 6, "axes.labelsize": 6,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
})

CM = 1 / 2.54


In [ ]:
from pathlib import Path

import caveclient
import numpy as np
import pandas as pd

# 669 is the newest version available when this repo was built. Two islet-cltmr neurons
# were split manually after it: prefer a LATER materialization and regenerate
# ../data/psi_ihn_compartments.csv with data_generation/build_psi_ihn_compartment_map.ipynb.
MATERIALIZATION = 669
SYNAPSE_TABLE = "synapses_version3"
AXOAXONIC_TABLE = "axoaxonic_output_synapses"
MIN_CLEFT_SIZE = 20

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")
if MATERIALIZATION is not None:
    client.version = MATERIALIZATION

compartments = pd.read_csv("../data/psi_ihn_compartments.csv")
print(compartments.tag.value_counts().to_string())

unsplit = compartments[~compartments.compartments_split]
if len(unsplit):
    print(f"\nWARNING: {len(unsplit)} of {len(compartments)} neurons have axon and "
          f"dendrite resolving to the same segment:")
    print(unsplit[["tag", "dend_coord", "axon_coord", "dend_root_id"]].to_string(index=False))


In [ ]:
def count_presynaptic(pre_root_id):
    """All synapses_version3 output of a segment: autapses out, cleft size >= 20."""
    df = client.materialize.query_table(
        SYNAPSE_TABLE, filter_equal_dict={"pre_pt_root_id": int(pre_root_id)})
    if df is None or df.empty:
        return 0
    df = df[(df.pre_pt_root_id != df.post_pt_root_id) & (df["size"] >= MIN_CLEFT_SIZE)]
    return int(len(df))


def count_axoaxonic(pre_root_id):
    """axoaxonic_output_synapses with this segment presynaptic; no further filtering."""
    df = client.materialize.query_table(
        AXOAXONIC_TABLE, filter_equal_dict={"pre_pt_root_id": int(pre_root_id)})
    return 0 if df is None else int(len(df))


records = []
for _, r in compartments.iterrows():
    axoaxonic = count_axoaxonic(r.axon_root_id)
    axon_total = count_presynaptic(r.axon_root_id)

    # Axoaxonic synapses also appear in synapses_version3, so the axodendritic
    # count is the remainder of the axon's output, not a separate query.
    axodendritic = max(axon_total - axoaxonic, 0)

    records.append({
        "tag": r.tag,
        "compartments_split": bool(r.compartments_split),
        "axoaxonic": axoaxonic,
        "axodendritic": axodendritic,
        "dendritic": count_presynaptic(r.dend_root_id),
    })

counts = pd.DataFrame(records)
counts["total"] = counts[["axoaxonic", "axodendritic", "dendritic"]].sum(axis=1)
counts = counts[counts.total > 0]
for c in ("axoaxonic", "axodendritic", "dendritic"):
    counts[f"frac_{c}"] = counts[c] / counts.total
print(counts.groupby("tag")[["frac_axoaxonic", "frac_axodendritic",
                             "frac_dendritic"]].mean().round(3).to_string())


In [ ]:
import matplotlib.pyplot as plt

TAGS_ORDER = ["ad-htmr", "islet-chtmr", "islet-cltmr", "islet-adltmr", "IN-abltmr"]
TAG_LABELS = {"ad-htmr": "PSI-IhN Aδ-HTMR", "islet-chtmr": "Islet C-HTMR/Heat",
              "islet-cltmr": "Islet C-LTMR", "islet-adltmr": "Islet Aδ-LTMR",
              "IN-abltmr": "PSI-IhN Aβ-LTMR"}
TYPES = [("axoaxonic", "Axo-axonic", "#f37420"),
         ("axodendritic", "Axo-dendritic", "#111314"),
         ("dendritic", "Dendritic", "#00978d")]


def plot_fractions(df, out_svg, title=""):
    fig, ax = plt.subplots(figsize=(5 * CM, 4 * CM))
    width = 0.26
    rng = np.random.default_rng(0)

    for j, (key, label, colour) in enumerate(TYPES):
        xs, means = [], []
        for i, tag in enumerate(TAGS_ORDER):
            vals = df.loc[df.tag == tag, f"frac_{key}"].to_numpy()
            x = i + (j - 1) * width
            xs.append(x)
            means.append(vals.mean() if len(vals) else 0.0)
            ax.scatter(x + rng.uniform(-0.05, 0.05, len(vals)), vals,
                       s=1.5, color="black", alpha=0.6, linewidths=0, zorder=3)
        ax.bar(xs, means, width=width, color=colour, label=label, zorder=1)

    ax.set_xticks(range(len(TAGS_ORDER)))
    ax.set_xticklabels([TAG_LABELS[t] for t in TAGS_ORDER],
                       rotation=-40, ha="left", fontsize=5)
    ax.set_ylabel("Syn Fraction")
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.0, 0.5, 1.0])
    if title:
        ax.set_title(title, fontsize=5)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.legend(frameon=False, fontsize=5, loc="upper right")
    fig.tight_layout()
    fig.savefig(out_svg, format="svg")
    plt.show()


plot_fractions(counts, "figure5b_synapse_type_fractions.svg")


In [ ]:
# Same panel excluding the neurons whose compartments are no longer split, where
# the axodendritic and dendritic counts are necessarily identical.
split_only = counts[counts.compartments_split]
if len(split_only) < len(counts):
    print(f"excluding {len(counts) - len(split_only)} unsplit neuron(s)\n")
    print(split_only.groupby("tag")[["frac_axoaxonic", "frac_axodendritic",
                                     "frac_dendritic"]].mean().round(3).to_string())
    plot_fractions(split_only, "figure5b_synapse_type_fractions_split_only.svg",
                   title="split compartments only")
else:
    print("all neurons have separate axon and dendrite segments")
